# PDF Parser API 테스트 및 활용

FastAPI 서버를 테스트하고 활용하는 노트북입니다.

## JupyterLab 서버 시작 방법

터미널에서 다음 명령어를 실행하여 JupyterLab 서버를 시작하세요:

```bash
# JupyterLab 서버 시작 (포트 8000)
uv run jupyter lab --ip=0.0.0.0 --port=8000 --no-browser
```

출력되는 URL과 토큰을 사용하여 브라우저에서 접속:
- **로컬**: `http://localhost:8000/?token=<your-token>`
- **원격 서버**: SSH 포트 포워딩 후 접속
  ```bash
  ssh -L 8000:localhost:8000 user@server
  ```

## 사전 준비

1. **FastAPI 서버**가 실행 중이어야 합니다:
   ```bash
   # 단일 워커 (개발용)
   uv run uvicorn api:app --host 0.0.0.0 --port 3000
   
   # 멀티 워커 (프로덕션, 동시 요청 처리)
   uv run uvicorn api:app --host 0.0.0.0 --port 3000 --workers 2
   ```

2. **AWS 자격증명**이 설정되어 있어야 합니다 (Bedrock 및 S3 접근):
   ```bash
   aws configure
   # 또는 환경 변수 설정
   export AWS_ACCESS_KEY_ID=your_key
   export AWS_SECRET_ACCESS_KEY=your_secret
   export AWS_DEFAULT_REGION=ap-northeast-2
   ```

**참고:**
- `--workers 2`: 2개의 독립적인 프로세스로 실행되어 동시에 2개의 요청을 처리할 수 있습니다
- c7i.large의 2코어를 활용하여 처리량 2배 증가

In [3]:
import requests
import json
import time
from pathlib import Path
from IPython.display import display, Markdown, HTML
import pandas as pd

In [4]:
# API 서버 URL
API_BASE_URL = "http://localhost:3000"

# Health check
response = requests.get(f"{API_BASE_URL}/health")
print(f"Status Code: {response.status_code}")
print(f"Response:")
print(json.dumps(response.json(), indent=2, ensure_ascii=False))

if response.status_code == 200:
    print("\n✅ API 서버가 정상적으로 실행 중입니다!")
    
    # 지원 형식 표시
    health_data = response.json()
    if "supported_formats" in health_data:
        print("\n📚 지원하는 파일 형식:")
        for category, formats in health_data["supported_formats"].items():
            print(f"   {category.upper():8s}: {', '.join(formats)}")
else:
    print("❌ API 서버에 접근할 수 없습니다.")

Status Code: 200
Response:
{
  "status": "healthy",
  "service": "document-parser-api",
  "version": "0.2.0",
  "supported_formats": {
    "pdf": [
      ".pdf"
    ],
    "office": [
      ".pptx",
      ".ods",
      ".xlsx",
      ".odt",
      ".rtf",
      ".odp",
      ".docx"
    ]
  }
}

✅ API 서버가 정상적으로 실행 중입니다!

📚 지원하는 파일 형식:
   PDF     : .pdf
   OFFICE  : .pptx, .ods, .xlsx, .odt, .rtf, .odp, .docx


In [5]:
# API 서버 URL
API_BASE_URL = "http://localhost:3000"

# Health check
response = requests.get(f"{API_BASE_URL}/health")
print(f"Status Code: {response.status_code}")
print(f"Response: {response.json()}")

if response.status_code == 200:
    print("✅ API 서버가 정상적으로 실행 중입니다!")
else:
    print("❌ API 서버에 접근할 수 없습니다.")

Status Code: 200
Response: {'status': 'healthy', 'service': 'document-parser-api', 'version': '0.2.0', 'supported_formats': {'pdf': ['.pdf'], 'office': ['.pptx', '.ods', '.xlsx', '.odt', '.rtf', '.odp', '.docx']}}
✅ API 서버가 정상적으로 실행 중입니다!


## 2. API 스키마 확인

OpenAPI 스키마를 확인하여 사용 가능한 엔드포인트를 살펴봅니다.

In [6]:
# OpenAPI 스키마 가져오기
schema = requests.get(f"{API_BASE_URL}/openapi.json").json()

print("📚 API 정보:")
print(f"  제목: {schema['info']['title']}")
print(f"  설명: {schema['info']['description']}")
print(f"  버전: {schema['info']['version']}")
print("\n📍 사용 가능한 엔드포인트:")
for path, methods in schema['paths'].items():
    for method, details in methods.items():
        print(f"  {method.upper():6s} {path:20s} - {details.get('summary', 'N/A')}")

📚 API 정보:
  제목: Document Parser API
  설명: Extract text, tables, and images from PDFs and Office documents with AI summaries
  버전: 0.2.0

📍 사용 가능한 엔드포인트:
  POST   /ocr                 - Ocr Pdf
  GET    /health              - Health Check
  POST   /process             - Process Pdf
  POST   /process-document    - Process Document
  POST   /process-office      - Process Office


## 3. PDF 처리 테스트

### 3.1 S3 경로 설정

In [7]:
# S3 경로 설정 (실제 경로로 변경하세요)
INPUT_PDF = "s3://miraeasset-product-knowledge-graph/zeroin/r3/R3_A0005C0_001.pdf"  # 처리할 PDF 경로
OUTPUT_BASE = "s3://miraeasset-product-knowledge-graph/output_md/"   

print(f"📄 입력 PDF: {INPUT_PDF}")
print(f"📂 출력 경로: {OUTPUT_BASE}")

📄 입력 PDF: s3://miraeasset-product-knowledge-graph/zeroin/r3/R3_A0005C0_001.pdf
📂 출력 경로: s3://miraeasset-product-knowledge-graph/output_md/


### 3.2 기본 처리 요청 (요약 포함)

In [8]:
# 요청 데이터 준비
request_data = {
    "inputPath": INPUT_PDF,
    "outputPath": OUTPUT_BASE,
    "modelId": "ap-northeast-2.anthropic.claude-haiku-4-5-20251001-v1:0",
    "noSummary": False,
    "tableMode": "accurate"
}

print("🚀 PDF 처리 요청 중...")
print(json.dumps(request_data, indent=2, ensure_ascii=False))

# API 호출
start_time = time.time()
response = requests.post(
    f"{API_BASE_URL}/process",
    json=request_data,
    timeout=600  # 10분 타임아웃
)
elapsed = time.time() - start_time

print(f"\n⏱️  요청 소요 시간: {elapsed:.2f}초")
print(f"📊 응답 코드: {response.status_code}")

if response.status_code == 200:
    result = response.json()
    print("\n✅ 처리 성공!")
    print(json.dumps(result, indent=2, ensure_ascii=False))
else:
    print(f"\n❌ 처리 실패: {response.status_code}")
    print(response.text)

🚀 PDF 처리 요청 중...
{
  "inputPath": "s3://miraeasset-product-knowledge-graph/zeroin/r3/R3_A0005C0_001.pdf",
  "outputPath": "s3://miraeasset-product-knowledge-graph/output_md/",
  "modelId": "ap-northeast-2.anthropic.claude-haiku-4-5-20251001-v1:0",
  "noSummary": false,
  "tableMode": "accurate"
}

⏱️  요청 소요 시간: 72.19초
📊 응답 코드: 200

✅ 처리 성공!
{
  "success": true,
  "message": "Successfully processed R3_A0005C0_001.pdf",
  "finalMarkdownUri": "s3://miraeasset-product-knowledge-graph/output_md/R3_A0005C0_001/R3_A0005C0_001_final.md",
  "uploadedFiles": [
    "s3://miraeasset-product-knowledge-graph/output_md/R3_A0005C0_001/R3_A0005C0_001_text.md",
    "s3://miraeasset-product-knowledge-graph/output_md/R3_A0005C0_001/R3_A0005C0_001_final.md",
    "s3://miraeasset-product-knowledge-graph/output_md/R3_A0005C0_001/pictures/table/R3_A0005C0_001_picture_1.png",
    "s3://miraeasset-product-knowledge-graph/output_md/R3_A0005C0_001/table/md/R3_A0005C0_001_table_7.md",
    "s3://miraeasset-product-

### 3.3 결과 분석 및 시각화

In [9]:
# 응답이 성공적인 경우에만 실행
if response.status_code == 200:
    result = response.json()
    
    # 기본 정보
    print("📋 처리 결과 요약")
    print("=" * 60)
    print(f"성공 여부: {result['success']}")
    print(f"메시지: {result['message']}")
    print(f"최종 마크다운: {result['finalMarkdownUri']}")
    print(f"소요 시간: {result['elapsedSeconds']}초")
    
    # 통계 정보
    if result.get('stats'):
        stats = result['stats']
        print("\n📊 추출 통계")
        print("=" * 60)
        print(f"페이지 수: {stats['pages']}")
        print(f"그림 수: {stats['figures']}")
        print(f"테이블 수: {stats['tables']}")
        print(f"요약 생성 여부: {stats['summariesGenerated']}")
        
        # DataFrame으로 시각화
        stats_df = pd.DataFrame([
            {"항목": "페이지", "개수": stats['pages']},
            {"항목": "그림", "개수": stats['figures']},
            {"항목": "테이블", "개수": stats['tables']},
        ])
        display(stats_df)
    
    # 업로드된 파일 목록
    if result.get('uploadedFiles'):
        print(f"\n📤 업로드된 파일 ({len(result['uploadedFiles'])}개)")
        print("=" * 60)
        for i, file_uri in enumerate(result['uploadedFiles'][:10], 1):
            print(f"{i:2d}. {file_uri}")
        if len(result['uploadedFiles']) > 10:
            print(f"    ... 외 {len(result['uploadedFiles']) - 10}개")

📋 처리 결과 요약
성공 여부: True
메시지: Successfully processed R3_A0005C0_001.pdf
최종 마크다운: s3://miraeasset-product-knowledge-graph/output_md/R3_A0005C0_001/R3_A0005C0_001_final.md
소요 시간: 72.18초

📊 추출 통계
페이지 수: 5
그림 수: 1
테이블 수: 8
요약 생성 여부: True


,항목,개수
0,페이지,5
1,그림,1
2,테이블,8



📤 업로드된 파일 (19개)
 1. s3://miraeasset-product-knowledge-graph/output_md/R3_A0005C0_001/R3_A0005C0_001_text.md
 2. s3://miraeasset-product-knowledge-graph/output_md/R3_A0005C0_001/R3_A0005C0_001_final.md
 3. s3://miraeasset-product-knowledge-graph/output_md/R3_A0005C0_001/pictures/table/R3_A0005C0_001_picture_1.png
 4. s3://miraeasset-product-knowledge-graph/output_md/R3_A0005C0_001/table/md/R3_A0005C0_001_table_7.md
 5. s3://miraeasset-product-knowledge-graph/output_md/R3_A0005C0_001/table/md/R3_A0005C0_001_table_6.md
 6. s3://miraeasset-product-knowledge-graph/output_md/R3_A0005C0_001/table/md/R3_A0005C0_001_table_1.md
 7. s3://miraeasset-product-knowledge-graph/output_md/R3_A0005C0_001/table/md/R3_A0005C0_001_table_4.md
 8. s3://miraeasset-product-knowledge-graph/output_md/R3_A0005C0_001/table/md/R3_A0005C0_001_table_2.md
 9. s3://miraeasset-product-knowledge-graph/output_md/R3_A0005C0_001/table/md/R3_A0005C0_001_table_3.md
10. s3://miraeasset-product-knowledge-graph/output_md/R3_A000

## 4. 다양한 옵션으로 처리 테스트

### 4.1 Fast 모드 (요약 없이, 빠른 테이블 추출)

In [7]:
# Fast 모드 요청
fast_request = {
    "inputPath": INPUT_PDF,
    "outputPath": OUTPUT_BASE,
    "noSummary": True,      # 요약 스킵
    "tableMode": "fast"     # 빠른 테이블 모드
}

print("⚡ Fast 모드 처리 중...")
start_time = time.time()
response = requests.post(f"{API_BASE_URL}/process", json=fast_request, timeout=600)
elapsed = time.time() - start_time

print(f"소요 시간: {elapsed:.2f}초")
if response.status_code == 200:
    result = response.json()
    print(f"✅ 성공: {result['message']}")
    print(f"서버 처리 시간: {result['elapsedSeconds']}초")
else:
    print(f"❌ 실패: {response.text}")

⚡ Fast 모드 처리 중...
소요 시간: 0.56초
❌ 실패: {"detail":"Processing failed: An error occurred (403) when calling the HeadObject operation: Forbidden"}


### 4.2 다른 Bedrock 모델 사용 (Sonnet 3.5)

In [8]:
# Sonnet 모델 사용
sonnet_request = {
    "inputPath": INPUT_PDF,
    "outputPath": OUTPUT_BASE,
    "modelId": "ap-northeast-2.anthropic.claude-3-5-sonnet-20241022-v2:0",  # Sonnet 모델
    "noSummary": False,
    "tableMode": "accurate"
}

print("🧠 Sonnet 모델로 처리 중...")
start_time = time.time()
response = requests.post(f"{API_BASE_URL}/process", json=sonnet_request, timeout=600)
elapsed = time.time() - start_time

print(f"소요 시간: {elapsed:.2f}초")
if response.status_code == 200:
    result = response.json()
    print(f"✅ 성공: {result['message']}")
    print(f"서버 처리 시간: {result['elapsedSeconds']}초")
else:
    print(f"❌ 실패: {response.text}")

🧠 Sonnet 모델로 처리 중...
소요 시간: 0.54초
❌ 실패: {"detail":"Processing failed: An error occurred (403) when calling the HeadObject operation: Forbidden"}


## 5. S3 결과 다운로드 및 확인

In [ ]:
# S3 결과 다운로드
from pdf_parser.s3_handler import S3Handler

if response.status_code == 200:
    result = response.json()
    final_md_uri = result['finalMarkdownUri']
    
    # S3에서 최종 마크다운 읽기
    s3 = S3Handler()
    
    print(f"📥 S3에서 마크다운 다운로드 중: {final_md_uri}")
    markdown_content = s3.read_markdown(final_md_uri)
    
    # 처음 1000자만 출력
    print("\n📄 마크다운 미리보기 (처음 1000자):")
    print("=" * 60)
    print(markdown_content[:1000])
    print("...")
    
    # 전체 길이
    print(f"\n📏 전체 길이: {len(markdown_content)} 문자")

### 5.1 전체 결과 폴더 다운로드

In [15]:
# 전체 결과 폴더를 로컬에 다운로드
if response.status_code == 200:
    result = response.json()
    final_md_uri = result['finalMarkdownUri']
    
    # 결과 폴더 URI 추출 (final.md 파일의 부모 디렉토리)
    result_folder_uri = "/".join(final_md_uri.split("/")[:-1]) + "/"
    
    # 로컬 다운로드 경로
    local_download_dir = Path("./downloads") / Path(INPUT_PDF).stem
    local_download_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"📥 S3 폴더 다운로드 중...")
    print(f"   From: {result_folder_uri}")
    print(f"   To:   {local_download_dir}")
    
    file_count = s3.download_directory(result_folder_uri, local_download_dir)
    
    print(f"\n✅ {file_count}개 파일 다운로드 완료!")
    print(f"\n📂 다운로드된 파일 목록:")
    for file_path in sorted(local_download_dir.rglob("*")):
        if file_path.is_file():
            size_kb = file_path.stat().st_size / 1024
            print(f"  {file_path.relative_to(local_download_dir)} ({size_kb:.1f} KB)")

📥 S3 폴더 다운로드 중...
   From: s3://miraeasset-product-knowledge-graph/output_md/R3_A0005C0_001/
   To:   downloads/R3_A0005C0_001

✅ 19개 파일 다운로드 완료!

📂 다운로드된 파일 목록:
  R3_A0005C0_001_final.md (101.4 KB)
  R3_A0005C0_001_text.md (93.7 KB)
  pictures/table/R3_A0005C0_001_picture_1.png (25.6 KB)
  table/img/R3_A0005C0_001_table_1.png (25.6 KB)
  table/img/R3_A0005C0_001_table_2.png (51.7 KB)
  table/img/R3_A0005C0_001_table_3.png (25.8 KB)
  table/img/R3_A0005C0_001_table_4.png (52.7 KB)
  table/img/R3_A0005C0_001_table_5.png (577.4 KB)
  table/img/R3_A0005C0_001_table_6.png (906.3 KB)
  table/img/R3_A0005C0_001_table_7.png (852.7 KB)
  table/img/R3_A0005C0_001_table_8.png (501.7 KB)
  table/md/R3_A0005C0_001_table_1.md (0.9 KB)
  table/md/R3_A0005C0_001_table_2.md (2.8 KB)
  table/md/R3_A0005C0_001_table_3.md (0.6 KB)
  table/md/R3_A0005C0_001_table_4.md (1.5 KB)
  table/md/R3_A0005C0_001_table_5.md (8.2 KB)
  table/md/R3_A0005C0_001_table_6.md (12.8 KB)
  table/md/R3_A0005C0_001_table_7.md 

## 6. 배치 처리 시뮬레이션

여러 PDF를 순차적으로 처리합니다.

In [11]:
# 처리할 PDF 목록 (실제 경로로 변경)
pdf_list = [
    "s3://your-bucket/input/doc1.pdf",
    "s3://your-bucket/input/doc2.pdf",
    "s3://your-bucket/input/doc3.pdf",
]

results = []

print(f"🔄 {len(pdf_list)}개 PDF 배치 처리 시작\n")

for i, pdf_uri in enumerate(pdf_list, 1):
    print(f"[{i}/{len(pdf_list)}] {pdf_uri}")
    
    request_data = {
        "inputPath": pdf_uri,
        "outputPath": OUTPUT_BASE,
        "noSummary": True,  # 빠른 처리를 위해 요약 스킵
        "tableMode": "fast"
    }
    
    try:
        start = time.time()
        resp = requests.post(f"{API_BASE_URL}/process", json=request_data, timeout=600)
        elapsed = time.time() - start
        
        if resp.status_code == 200:
            result = resp.json()
            results.append({
                "PDF": Path(pdf_uri).name,
                "상태": "✅ 성공",
                "페이지": result['stats']['pages'],
                "그림": result['stats']['figures'],
                "테이블": result['stats']['tables'],
                "소요시간(초)": result['elapsedSeconds']
            })
            print(f"  ✅ 성공 ({result['elapsedSeconds']:.1f}초)\n")
        else:
            results.append({
                "PDF": Path(pdf_uri).name,
                "상태": f"❌ 실패 ({resp.status_code})",
                "페이지": None,
                "그림": None,
                "테이블": None,
                "소요시간(초)": elapsed
            })
            print(f"  ❌ 실패: {resp.status_code}\n")
    
    except Exception as e:
        results.append({
            "PDF": Path(pdf_uri).name,
            "상태": f"❌ 오류",
            "페이지": None,
            "그림": None,
            "테이블": None,
            "소요시간(초)": None
        })
        print(f"  ❌ 오류: {e}\n")

# 결과 요약
print("\n" + "=" * 80)
print("📊 배치 처리 결과 요약")
print("=" * 80)
results_df = pd.DataFrame(results)
display(results_df)

# 성공/실패 통계
success_count = sum(1 for r in results if "성공" in r["상태"])
fail_count = len(results) - success_count
print(f"\n✅ 성공: {success_count}개")
print(f"❌ 실패: {fail_count}개")

🔄 3개 PDF 배치 처리 시작

[1/3] s3://your-bucket/input/doc1.pdf
  ❌ 실패: 500

[2/3] s3://your-bucket/input/doc2.pdf
  ❌ 실패: 500

[3/3] s3://your-bucket/input/doc3.pdf
  ❌ 실패: 500


📊 배치 처리 결과 요약


,PDF,상태,페이지,그림,테이블,소요시간(초)
0,doc1.pdf,❌ 실패 (500),None,None,None,0.555468
1,doc2.pdf,❌ 실패 (500),None,None,None,0.570723
2,doc3.pdf,❌ 실패 (500),None,None,None,0.564170



✅ 성공: 0개
❌ 실패: 3개


## 7. 에러 핸들링 테스트

In [12]:
# 잘못된 요청 테스트
print("🧪 에러 핸들링 테스트\n")

# 1. 잘못된 S3 URI (http://)
print("1. 잘못된 S3 URI 테스트")
bad_request = {"inputPath": "http://example.com/file.pdf", "outputPath": OUTPUT_BASE}
resp = requests.post(f"{API_BASE_URL}/process", json=bad_request)
print(f"   응답 코드: {resp.status_code}")
print(f"   응답 메시지: {resp.json().get('detail', resp.text)}\n")

# 2. PDF가 아닌 파일
print("2. PDF가 아닌 파일 테스트")
bad_request = {"inputPath": "s3://bucket/file.txt", "outputPath": OUTPUT_BASE}
resp = requests.post(f"{API_BASE_URL}/process", json=bad_request)
print(f"   응답 코드: {resp.status_code}")
print(f"   응답 메시지: {resp.json().get('detail', resp.text)}\n")

# 3. 잘못된 tableMode
print("3. 잘못된 tableMode 테스트")
bad_request = {"inputPath": INPUT_PDF, "outputPath": OUTPUT_BASE, "tableMode": "invalid"}
resp = requests.post(f"{API_BASE_URL}/process", json=bad_request)
print(f"   응답 코드: {resp.status_code}")
print(f"   응답 메시지: {resp.json().get('detail', resp.text)}\n")

print("✅ 모든 에러가 적절히 처리되었습니다!")

🧪 에러 핸들링 테스트

1. 잘못된 S3 URI 테스트
   응답 코드: 400
   응답 메시지: inputPath must be S3 URI starting with s3://

2. PDF가 아닌 파일 테스트
   응답 코드: 400
   응답 메시지: inputPath must be a PDF file (ending with .pdf)

3. 잘못된 tableMode 테스트
   응답 코드: 400
   응답 메시지: tableMode must be 'accurate' or 'fast'

✅ 모든 에러가 적절히 처리되었습니다!


## 8. 성능 벤치마크

In [13]:
# Fast vs Accurate 모드 비교
import matplotlib.pyplot as plt

modes = ["fast", "accurate"]
mode_times = []

print("⏱️  Fast vs Accurate 모드 성능 비교\n")

for mode in modes:
    print(f"테스트 중: {mode} 모드")
    request_data = {
        "inputPath": INPUT_PDF,
        "outputPath": OUTPUT_BASE,
        "noSummary": True,
        "tableMode": mode
    }
    
    start = time.time()
    resp = requests.post(f"{API_BASE_URL}/process", json=request_data, timeout=600)
    elapsed = time.time() - start
    
    if resp.status_code == 200:
        result = resp.json()
        mode_times.append(result['elapsedSeconds'])
        print(f"  {mode:8s}: {result['elapsedSeconds']:.2f}초\n")
    else:
        print(f"  실패: {resp.status_code}\n")

# 시각화
if len(mode_times) == 2:
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(modes, mode_times, color=['#FF6B6B', '#4ECDC4'])
    ax.set_ylabel('처리 시간 (초)')
    ax.set_title('Table Mode 성능 비교')
    ax.set_ylim(0, max(mode_times) * 1.2)
    
    # 막대 위에 값 표시
    for bar, time_val in zip(bars, mode_times):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{time_val:.2f}초',
                ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    # 속도 향상 비율
    speedup = (mode_times[1] - mode_times[0]) / mode_times[1] * 100
    print(f"\n⚡ Fast 모드가 Accurate 모드보다 {speedup:.1f}% 빠릅니다!")

ModuleNotFoundError: No module named 'matplotlib'

## 9. 유틸리티 함수

자주 사용하는 작업을 함수로 만듭니다.

In [ ]:
def process_pdf_simple(input_uri: str, output_base: str, fast_mode: bool = True):
    """간단한 PDF 처리 함수"""
    request_data = {
        "inputPath": input_uri,
        "outputPath": output_base,
        "noSummary": fast_mode,
        "tableMode": "fast" if fast_mode else "accurate"
    }
    
    resp = requests.post(f"{API_BASE_URL}/process", json=request_data, timeout=600)
    
    if resp.status_code == 200:
        return resp.json()
    else:
        raise Exception(f"API Error: {resp.status_code} - {resp.text}")


def get_processing_stats(result: dict):
    """처리 결과에서 통계 추출"""
    stats = result.get('stats', {})
    return {
        "페이지": stats.get('pages', 0),
        "그림": stats.get('figures', 0),
        "테이블": stats.get('tables', 0),
        "소요시간(초)": result.get('elapsedSeconds', 0),
        "최종파일": result.get('finalMarkdownUri', '')
    }


# 사용 예시
print("📦 유틸리티 함수 정의 완료!")
print("\n사용 예시:")
print('result = process_pdf_simple("s3://bucket/file.pdf", "s3://bucket/output/")')
print('stats = get_processing_stats(result)')

## 10. 요약

이 노트북에서는 다음을 다루었습니다:

1. ✅ API Health Check
2. ✅ OpenAPI 스키마 확인
3. ✅ 기본 PDF 처리
4. ✅ 다양한 옵션 (Fast/Accurate, 모델 변경)
5. ✅ S3 결과 다운로드
6. ✅ 배치 처리
7. ✅ 에러 핸들링
8. ✅ 성능 벤치마크
9. ✅ 유틸리티 함수

### API 문서 링크

- Swagger UI: http://localhost:3000/docs
- ReDoc: http://localhost:3000/redoc
- OpenAPI JSON: http://localhost:3000/openapi.json

## 11. S3 파일 브라우저

JupyterLab에서 인터랙티브하게 S3 경로를 탐색하고 PDF를 선택할 수 있는 파일 브라우저입니다.

In [ ]:
# S3 브라우저 임포트 및 생성
from pdf_parser.s3_browser import create_s3_browser

# 초기 경로 설정 (실제 S3 경로로 변경)
INITIAL_S3_PATH = "s3://miraeasset-product-knowledge-graph/"

# 브라우저 생성 및 표시
browser = create_s3_browser(initial_path=INITIAL_S3_PATH)

print("💡 사용 방법:")
print("  1. 폴더를 클릭하여 하위 폴더로 이동")
print("  2. PDF 파일을 클릭하여 선택")
print("  3. '↑ Parent' 버튼으로 상위 폴더로 이동")
print("  4. 경로를 직접 입력하고 'Go' 버튼 클릭")
print("  5. '↻ Refresh' 버튼으로 현재 경로 새로고침")

### 11.1 선택된 파일 확인 및 처리

In [ ]:
# 선택된 PDF 파일 가져오기
selected_pdf = browser.get_selected()

if selected_pdf:
    print(f"✅ 선택된 파일: {selected_pdf}")
    
    # API를 통해 처리 실행
    print("\n🚀 처리 시작...")
    
    request_data = {
        "inputPath": selected_pdf,
        "outputPath": OUTPUT_BASE,
        "modelId": "ap-northeast-2.anthropic.claude-haiku-4-5-20251001-v1:0",
        "noSummary": False,
        "tableMode": "accurate"
    }
    
    start_time = time.time()
    response = requests.post(
        f"{API_BASE_URL}/process",
        json=request_data,
        timeout=600
    )
    elapsed = time.time() - start_time
    
    print(f"⏱️  소요 시간: {elapsed:.2f}초")
    
    if response.status_code == 200:
        result = response.json()
        print(f"\n✅ 처리 완료!")
        print(f"   최종 마크다운: {result['finalMarkdownUri']}")
        print(f"   페이지: {result['stats']['pages']}")
        print(f"   그림: {result['stats']['figures']}")
        print(f"   테이블: {result['stats']['tables']}")
    else:
        print(f"\n❌ 처리 실패: {response.status_code}")
        print(f"   {response.text}")
else:
    print("⚠️  파일을 먼저 선택해주세요.")

### 11.2 콜백 함수를 사용한 자동 처리

PDF를 선택하면 자동으로 처리가 시작되도록 콜백 함수를 설정할 수 있습니다.

In [ ]:
# 콜백 함수 정의
def on_pdf_selected(s3_uri: str):
    """PDF가 선택되면 자동으로 실행되는 함수"""
    print(f"\n📄 선택됨: {s3_uri}")
    print("=" * 80)
    
    # 자동으로 처리할지 물어보기 (실제로는 바로 처리)
    print("💡 Tip: 이 셀 아래에서 선택된 파일로 처리를 실행할 수 있습니다.")

# 콜백이 있는 브라우저 생성
browser_with_callback = create_s3_browser(
    initial_path=INITIAL_S3_PATH,
    on_select=on_pdf_selected
)

print("✨ PDF를 클릭하면 자동으로 선택 정보가 출력됩니다!")

In [ ]:
# 간단한 속도 테스트 함수
def quick_speed_test(pdf_uri: str, output_uri: str):
    """두 가지 설정(기본 vs 최적화)의 속도를 빠르게 비교"""
    
    configs = [
        {"name": "기본 설정", "tableMode": "accurate", "useAccelerator": False},
        {"name": "최적화 설정", "tableMode": "fast", "useAccelerator": True},
    ]
    
    print(f"🏁 빠른 속도 테스트: {Path(pdf_uri).name}")
    print("=" * 60)
    
    times = []
    for config in configs:
        request_data = {
            "inputPath": pdf_uri,
            "outputPath": output_uri,
            "tableMode": config["tableMode"],
            "generateBboxImages": False,
            "useAccelerator": config["useAccelerator"]
        }
        
        print(f"\n{config['name']} 테스트 중...")
        
        try:
            response = requests.post(f"{API_BASE_URL}/ocr", json=request_data, timeout=600)
            
            if response.status_code == 200:
                result = response.json()
                elapsed = result['elapsedSeconds']
                times.append(elapsed)
                print(f"  ⏱️  {elapsed:.2f}초")
            else:
                print(f"  ❌ 실패: {response.status_code}")
                times.append(None)
        except Exception as e:
            print(f"  ❌ 오류: {e}")
            times.append(None)
    
    # 결과 비교
    if all(t is not None for t in times):
        speedup = times[0] / times[1]
        improvement_pct = ((times[0] - times[1]) / times[0]) * 100
        
        print("\n" + "=" * 60)
        print(f"⚡ 최적화 설정이 {speedup:.2f}x 빠릅니다!")
        print(f"   {times[0]:.2f}초 → {times[1]:.2f}초 ({improvement_pct:.1f}% 개선)")
    
    return times

# 사용 예시 (실제 테스트를 실행하려면 주석 해제)
# quick_speed_test(INPUT_PDF, OUTPUT_BASE)

### 12.1 성능 최적화 권장사항

위 테스트 결과를 바탕으로 한 권장사항:

**1. 빠른 처리가 필요한 경우 (프로덕션)**
```json
{
  "tableMode": "fast",
  "useAccelerator": true
}
```
- 가장 빠른 처리 속도
- 테이블 정확도는 약간 낮아질 수 있음

**2. 정확도가 중요한 경우 (분석/보고서)**
```json
{
  "tableMode": "accurate",
  "useAccelerator": true
}
```
- 높은 테이블 정확도 유지
- Accelerator로 속도 개선

**3. 리소스 제한 환경**
```json
{
  "tableMode": "fast",
  "useAccelerator": false
}
```
- CPU 사용량 제한 시
- 메모리 제약이 있는 환경

**추가 최적화 방법:**
- **Uvicorn Workers 증가**: `--workers 2` (c7i.large의 2코어 활용)
- **배치 처리**: 여러 PDF를 병렬로 처리
- **OCR 전용 모드**: 요약이 필요없으면 `/ocr` 엔드포인트 사용

In [ ]:
# 결과 DataFrame으로 표시
results_df = pd.DataFrame(results)
display(results_df)

# 성공한 테스트만 필터링
success_results = [r for r in results if r["처리시간(초)"] is not None]

if len(success_results) > 0:
    print("\n" + "=" * 80)
    print("⚡ 성능 비교 분석")
    print("=" * 80)
    
    # 기준값 (Accurate + Accelerator OFF)
    baseline = None
    for r in success_results:
        if r["Table Mode"] == "accurate" and r["Accelerator"] == "❌ OFF":
            baseline = r["처리시간(초)"]
            break
    
    if baseline:
        print(f"\n📊 기준: Accurate + Accelerator OFF = {baseline:.2f}초\n")
        
        for r in success_results:
            time_val = r["처리시간(초)"]
            speedup = (baseline / time_val) if time_val > 0 else 0
            speedup_pct = ((baseline - time_val) / baseline * 100) if baseline > 0 else 0
            
            if speedup > 1:
                print(f"   {r['설정']:30s}: {time_val:6.2f}초 (🚀 {speedup:.2f}x 빠름, {speedup_pct:+.1f}%)")
            elif speedup < 1:
                print(f"   {r['설정']:30s}: {time_val:6.2f}초 (🐌 {1/speedup:.2f}x 느림, {speedup_pct:+.1f}%)")
            else:
                print(f"   {r['설정']:30s}: {time_val:6.2f}초 (기준)")
        
        # 가장 빠른 조합 찾기
        fastest = min(success_results, key=lambda x: x["처리시간(초)"])
        fastest_speedup = baseline / fastest["처리시간(초)"]
        
        print(f"\n🏆 최고 성능: {fastest['설정']}")
        print(f"   {fastest['처리시간(초)']:.2f}초 ({fastest_speedup:.2f}x 빠름)")
    
    # 간단한 막대 그래프 (텍스트 기반)
    print("\n" + "=" * 80)
    print("📈 처리 시간 비교 (막대 그래프)")
    print("=" * 80)
    
    max_time = max(r["처리시간(초)"] for r in success_results)
    bar_width = 60  # 최대 막대 길이
    
    for r in success_results:
        time_val = r["처리시간(초)"]
        bar_len = int((time_val / max_time) * bar_width)
        bar = "█" * bar_len
        
        config_short = r['설정'].replace(" + Accelerator", "").replace("Accurate", "Acc").replace("Fast", "Fst")
        acc_mark = "⚡" if r["Accelerator"] == "✅ ON" else "  "
        
        print(f"{acc_mark} {config_short:20s} {bar} {time_val:.2f}s")
else:
    print("\n⚠️  성공한 테스트가 없어 비교 분석을 수행할 수 없습니다.")

In [ ]:
# 속도 테스트를 위한 설정
TEST_PDF = INPUT_PDF  # 이미 설정된 PDF 경로 사용
TEST_OUTPUT = OUTPUT_BASE

# 테스트 조합
test_configs = [
    {"name": "Accurate + Accelerator OFF", "tableMode": "accurate", "useAccelerator": False},
    {"name": "Accurate + Accelerator ON", "tableMode": "accurate", "useAccelerator": True},
    {"name": "Fast + Accelerator OFF", "tableMode": "fast", "useAccelerator": False},
    {"name": "Fast + Accelerator ON", "tableMode": "fast", "useAccelerator": True},
]

results = []

print("🚀 Accelerator 성능 테스트 시작")
print("=" * 80)
print(f"테스트 PDF: {TEST_PDF}")
print(f"출력 경로: {TEST_OUTPUT}")
print(f"엔드포인트: /ocr (OCR 전용, 빠른 테스트)\n")

for config in test_configs:
    print(f"🧪 {config['name']}")
    print(f"   tableMode: {config['tableMode']}, useAccelerator: {config['useAccelerator']}")
    
    request_data = {
        "inputPath": TEST_PDF,
        "outputPath": TEST_OUTPUT,
        "tableMode": config["tableMode"],
        "generateBboxImages": False,  # 속도 테스트를 위해 bbox 이미지 생성 스킵
        "useAccelerator": config["useAccelerator"]
    }
    
    try:
        start_time = time.time()
        response = requests.post(
            f"{API_BASE_URL}/ocr",
            json=request_data,
            timeout=600
        )
        client_elapsed = time.time() - start_time
        
        if response.status_code == 200:
            result = response.json()
            server_elapsed = result['elapsedSeconds']
            stats = result['stats']
            
            results.append({
                "설정": config['name'],
                "Table Mode": config['tableMode'],
                "Accelerator": "✅ ON" if config['useAccelerator'] else "❌ OFF",
                "페이지": stats['pages'],
                "그림": stats['figures'],
                "테이블": stats['tables'],
                "처리시간(초)": server_elapsed,
                "상태": "✅ 성공"
            })
            
            print(f"   ✅ 완료: {server_elapsed:.2f}초\n")
        else:
            results.append({
                "설정": config['name'],
                "Table Mode": config['tableMode'],
                "Accelerator": "✅ ON" if config['useAccelerator'] else "❌ OFF",
                "페이지": None,
                "그림": None,
                "테이블": None,
                "처리시간(초)": None,
                "상태": f"❌ 실패 ({response.status_code})"
            })
            print(f"   ❌ 실패: {response.status_code} - {response.text}\n")
    
    except Exception as e:
        results.append({
            "설정": config['name'],
            "Table Mode": config['tableMode'],
            "Accelerator": "✅ ON" if config['useAccelerator'] else "❌ OFF",
            "페이지": None,
            "그림": None,
            "테이블": None,
            "처리시간(초)": None,
            "상태": f"❌ 오류"
        })
        print(f"   ❌ 오류: {e}\n")

print("\n" + "=" * 80)
print("📊 성능 테스트 결과")
print("=" * 80)

## 12. Accelerator 성능 테스트

CPU Accelerator 옵션을 켜고 끄면서 처리 속도를 비교합니다.

**테스트 조합:**
1. `accurate` + `accelerator OFF` (기본)
2. `accurate` + `accelerator ON`
3. `fast` + `accelerator OFF`
4. `fast` + `accelerator ON` (최고 속도)

## 13. Office 문서 처리 테스트

API v0.2.0에서 추가된 Office 문서 처리 기능을 테스트합니다.

**지원 형식:**
- Word: `.docx`
- PowerPoint: `.pptx`
- Excel: `.xlsx`
- OpenDocument: `.odt`, `.odp`, `.ods`
- RTF: `.rtf`

**새로운 엔드포인트:**
- `/process-office`: Office 문서 전용
- `/process-document`: PDF/Office 자동 구분 통합 엔드포인트

### 13.1 Office 문서 S3 경로 설정

In [ ]:
# Office 문서 S3 경로 설정 (실제 경로로 변경하세요)
INPUT_DOCX = "s3://your-bucket/input/report.docx"  # Word 문서
INPUT_PPTX = "s3://your-bucket/input/presentation.pptx"  # PowerPoint
INPUT_XLSX = "s3://your-bucket/input/spreadsheet.xlsx"  # Excel
OUTPUT_OFFICE_BASE = "s3://your-bucket/output-office/"  # Office 결과 저장 경로

print("📚 Office 문서 입력 경로:")
print(f"  Word:       {INPUT_DOCX}")
print(f"  PowerPoint: {INPUT_PPTX}")
print(f"  Excel:      {INPUT_XLSX}")
print(f"\n📂 출력 경로: {OUTPUT_OFFICE_BASE}")

### 13.2 Word 문서 처리 (Markdown 출력)

In [ ]:
# Word 문서 처리 요청 (Markdown 출력)
request_data = {
    "inputPath": INPUT_DOCX,
    "outputPath": OUTPUT_OFFICE_BASE,
    "modelId": "us.anthropic.claude-haiku-4-5-20251001-v1:0",
    "noSummary": False,
    "outputFormat": "markdown",
    "bedrockRegion": "us-east-1"
}

print("📝 Word 문서 처리 요청 중...")
print(json.dumps(request_data, indent=2, ensure_ascii=False))

start_time = time.time()
response = requests.post(
    f"{API_BASE_URL}/process-office",
    json=request_data,
    timeout=600
)
elapsed = time.time() - start_time

print(f"\n⏱️  요청 소요 시간: {elapsed:.2f}초")
print(f"📊 응답 코드: {response.status_code}")

if response.status_code == 200:
    result = response.json()
    print("\n✅ 처리 성공!")
    print(json.dumps(result, indent=2, ensure_ascii=False))
else:
    print(f"\n❌ 처리 실패: {response.status_code}")
    print(response.text)

### 13.3 통합 엔드포인트 테스트 (/process-document)

파일 확장자에 따라 자동으로 PDF 또는 Office 프로세서로 라우팅됩니다.

In [ ]:
# 통합 엔드포인트로 Word 문서 처리
request_data = {
    "inputPath": INPUT_DOCX,
    "outputPath": OUTPUT_OFFICE_BASE,
    "modelId": "us.anthropic.claude-haiku-4-5-20251001-v1:0",
    "noSummary": False,
    "outputFormat": "html",  # HTML 출력 테스트
    "bedrockRegion": "us-east-1"
}

print("🔀 통합 엔드포인트로 Word 문서 처리 (HTML 출력)")
print(f"입력: {request_data['inputPath']}")

start_time = time.time()
response = requests.post(
    f"{API_BASE_URL}/process-document",
    json=request_data,
    timeout=600
)
elapsed = time.time() - start_time

print(f"\n⏱️  소요 시간: {elapsed:.2f}초")

if response.status_code == 200:
    result = response.json()
    print("\n✅ 처리 성공!")
    print(f"   출력 파일: {result.get('outputUri')}")
    print(f"   파일 형식: {result['stats']['fileType']}")
    print(f"   출력 형식: {result['stats']['outputFormat']}")
    print(f"   첨부파일: {result['stats']['attachments']}개")
else:
    print(f"\n❌ 처리 실패: {response.status_code}")
    print(response.text)

### 13.4 다양한 출력 형식 테스트

In [ ]:
# 다양한 출력 형식으로 테스트
output_formats = ["markdown", "html", "text"]
format_results = []

print("🎨 출력 형식 비교 테스트\n")

for fmt in output_formats:
    print(f"테스트 중: {fmt.upper()} 형식")
    
    request_data = {
        "inputPath": INPUT_DOCX,
        "outputPath": OUTPUT_OFFICE_BASE,
        "outputFormat": fmt,
        "noSummary": True  # 빠른 테스트를 위해 요약 스킵
    }
    
    try:
        start = time.time()
        resp = requests.post(
            f"{API_BASE_URL}/process-office",
            json=request_data,
            timeout=600
        )
        elapsed = time.time() - start
        
        if resp.status_code == 200:
            result = resp.json()
            format_results.append({
                "형식": fmt.upper(),
                "상태": "✅ 성공",
                "소요시간(초)": result['elapsedSeconds'],
                "출력 URI": result['outputUri']
            })
            print(f"  ✅ 성공 ({result['elapsedSeconds']:.2f}초)\n")
        else:
            format_results.append({
                "형식": fmt.upper(),
                "상태": f"❌ 실패 ({resp.status_code})",
                "소요시간(초)": elapsed,
                "출력 URI": None
            })
            print(f"  ❌ 실패: {resp.status_code}\n")
    
    except Exception as e:
        format_results.append({
            "형식": fmt.upper(),
            "상태": "❌ 오류",
            "소요시간(초)": None,
            "출력 URI": None
        })
        print(f"  ❌ 오류: {e}\n")

# 결과 요약
print("\n" + "=" * 80)
print("📊 출력 형식 비교 결과")
print("=" * 80)
results_df = pd.DataFrame(format_results)
display(results_df)

### 13.5 PDF vs Office 문서 비교 처리

In [ ]:
# PDF와 Office 문서를 통합 엔드포인트로 처리하여 비교
test_files = [
    {"type": "PDF", "path": INPUT_PDF, "icon": "📄"},
    {"type": "Word", "path": INPUT_DOCX, "icon": "📝"},
    {"type": "PowerPoint", "path": INPUT_PPTX, "icon": "📊"},
    {"type": "Excel", "path": INPUT_XLSX, "icon": "📈"},
]

comparison_results = []

print("🔍 통합 엔드포인트 파일 형식 자동 인식 테스트\n")

for file_info in test_files:
    print(f"{file_info['icon']} {file_info['type']} 처리 중: {Path(file_info['path']).name}")
    
    # 동일한 요청 구조로 호출
    request_data = {
        "inputPath": file_info['path'],
        "outputPath": OUTPUT_OFFICE_BASE if file_info['type'] != "PDF" else OUTPUT_BASE,
        "noSummary": True  # 빠른 테스트
    }
    
    try:
        start = time.time()
        resp = requests.post(
            f"{API_BASE_URL}/process-document",  # 통합 엔드포인트 사용
            json=request_data,
            timeout=600
        )
        elapsed = time.time() - start
        
        if resp.status_code == 200:
            result = resp.json()
            
            # PDF와 Office의 응답 형식이 다를 수 있으므로 처리
            if 'finalMarkdownUri' in result:  # PDF 응답
                output_uri = result['finalMarkdownUri']
                stats = result['stats']
                comparison_results.append({
                    "파일 형식": file_info['type'],
                    "파일명": Path(file_info['path']).name,
                    "상태": "✅ 성공",
                    "페이지/시트": stats.get('pages', 'N/A'),
                    "그림": stats.get('figures', 'N/A'),
                    "테이블": stats.get('tables', 'N/A'),
                    "소요시간(초)": result['elapsedSeconds']
                })
            elif 'outputUri' in result:  # Office 응답
                output_uri = result['outputUri']
                stats = result['stats']
                comparison_results.append({
                    "파일 형식": file_info['type'],
                    "파일명": Path(file_info['path']).name,
                    "상태": "✅ 성공",
                    "페이지/시트": 'N/A',
                    "그림": stats.get('attachments', 'N/A'),
                    "테이블": 'N/A',
                    "소요시간(초)": result['elapsedSeconds']
                })
            
            print(f"  ✅ 성공 ({result['elapsedSeconds']:.2f}초)\n")
        else:
            comparison_results.append({
                "파일 형식": file_info['type'],
                "파일명": Path(file_info['path']).name,
                "상태": f"❌ 실패 ({resp.status_code})",
                "페이지/시트": None,
                "그림": None,
                "테이블": None,
                "소요시간(초)": elapsed
            })
            print(f"  ❌ 실패: {resp.status_code}\n")
    
    except Exception as e:
        comparison_results.append({
            "파일 형식": file_info['type'],
            "파일명": Path(file_info['path']).name,
            "상태": "❌ 오류",
            "페이지/시트": None,
            "그림": None,
            "테이블": None,
            "소요시간(초)": None
        })
        print(f"  ❌ 오류: {e}\n")

# 결과 비교 표시
print("\n" + "=" * 80)
print("📊 파일 형식별 처리 결과 비교")
print("=" * 80)
comparison_df = pd.DataFrame(comparison_results)
display(comparison_df)

# 성공률 통계
success_count = sum(1 for r in comparison_results if "성공" in r["상태"])
total_count = len(comparison_results)
print(f"\n✅ 성공률: {success_count}/{total_count} ({success_count/total_count*100:.1f}%)")

### 13.6 Office 문서 에러 핸들링 테스트

In [ ]:
# Office 문서 전용 에러 핸들링 테스트
print("🧪 Office 문서 에러 핸들링 테스트\n")

# 1. 잘못된 outputFormat
print("1. 잘못된 outputFormat 테스트")
bad_request = {
    "inputPath": INPUT_DOCX,
    "outputPath": OUTPUT_OFFICE_BASE,
    "outputFormat": "pdf"  # 지원하지 않는 형식
}
resp = requests.post(f"{API_BASE_URL}/process-office", json=bad_request)
print(f"   응답 코드: {resp.status_code}")
print(f"   응답 메시지: {resp.json().get('detail', resp.text)}\n")

# 2. 지원하지 않는 파일 형식
print("2. 지원하지 않는 파일 형식 테스트")
bad_request = {
    "inputPath": "s3://bucket/file.txt",
    "outputPath": OUTPUT_OFFICE_BASE
}
resp = requests.post(f"{API_BASE_URL}/process-office", json=bad_request)
print(f"   응답 코드: {resp.status_code}")
print(f"   응답 메시지: {resp.json().get('detail', resp.text)}\n")

# 3. 통합 엔드포인트에 지원하지 않는 형식
print("3. 통합 엔드포인트 지원하지 않는 형식 테스트")
bad_request = {
    "inputPath": "s3://bucket/file.csv",
    "outputPath": OUTPUT_OFFICE_BASE
}
resp = requests.post(f"{API_BASE_URL}/process-document", json=bad_request)
print(f"   응답 코드: {resp.status_code}")
print(f"   응답 메시지: {resp.json().get('detail', resp.text)}\n")

print("✅ 모든 에러가 적절히 처리되었습니다!")

### 13.7 Office 문서 처리 결과 다운로드 및 확인

In [ ]:
# Office 문서 처리 결과 다운로드 (성공한 경우만)
if response.status_code == 200 and 'outputUri' in result:
    output_uri = result['outputUri']
    
    # 결과 폴더 URI 추출
    result_folder_uri = "/".join(output_uri.split("/")[:-1]) + "/"
    
    # 로컬 다운로드 경로
    file_name = Path(INPUT_DOCX).stem
    local_download_dir = Path("./downloads_office") / file_name
    local_download_dir.mkdir(parents=True, exist_ok=True)
    
    from pdf_parser.s3_handler import S3Handler
    s3 = S3Handler()
    
    print(f"📥 Office 결과 다운로드 중...")
    print(f"   From: {result_folder_uri}")
    print(f"   To:   {local_download_dir}")
    
    file_count = s3.download_directory(result_folder_uri, local_download_dir)
    
    print(f"\n✅ {file_count}개 파일 다운로드 완료!")
    print(f"\n📂 다운로드된 파일 목록:")
    for file_path in sorted(local_download_dir.rglob("*")):
        if file_path.is_file():
            size_kb = file_path.stat().st_size / 1024
            print(f"  {file_path.relative_to(local_download_dir)} ({size_kb:.1f} KB)")
    
    # 출력 파일 미리보기
    output_file = local_download_dir / Path(output_uri).name
    if output_file.exists():
        content = output_file.read_text(encoding="utf-8")
        print(f"\n📄 출력 파일 미리보기 (처음 500자):")
        print("=" * 60)
        print(content[:500])
        print("...")
        print(f"\n📏 전체 길이: {len(content)} 문자")
else:
    print("⚠️  먼저 Office 문서 처리를 성공적으로 완료해주세요.")

## 14. Office 문서 처리 요약

이 섹션에서 테스트한 내용:

1. ✅ Office 문서 전용 엔드포인트 (`/process-office`)
2. ✅ 통합 엔드포인트 (`/process-document`) - 자동 라우팅
3. ✅ 다양한 출력 형식 (Markdown, HTML, Text)
4. ✅ PDF vs Office 문서 비교 처리
5. ✅ Office 전용 에러 핸들링
6. ✅ 결과 다운로드 및 확인

### 주요 차이점

| 항목 | PDF | Office |
|------|-----|--------|
| 엔드포인트 | `/process`, `/ocr` | `/process-office` |
| 통합 엔드포인트 | ✅ `/process-document` | ✅ `/process-document` |
| 출력 형식 | Markdown (고정) | Markdown, HTML, Text |
| 테이블 모드 | Accurate, Fast | N/A |
| Accelerator | ✅ 지원 | ❌ 미지원 |
| 첨부파일 | 자동 추출 | pictures/ 폴더에 저장 |

### 권장 사용 방법

**1. 통합 엔드포인트 사용 (권장)**
```python
# 파일 형식에 관계없이 동일한 방식으로 처리
POST /process-document
```

**2. 전용 엔드포인트 사용**
```python
# PDF 전용
POST /process

# Office 전용
POST /process-office
```

**3. 출력 형식 선택**
- `markdown`: 일반적인 문서 변환 (기본값)
- `html`: 웹 표시용, 메타데이터 테이블 포함
- `text`: 순수 텍스트만 필요한 경우